# A0 — 방향 A 여지(headroom) 검증

**질문:** 샘플마다 다른 행동(no-op / retain / accept, early / late)을 고를 수 있다면, 동결 설정의 Always-retain보다 얼마나 좋아질 수 있는가?

- 기준 설정: Steering_develop 동결 설정 (p8s-v2 C→Q, P9 전체 토큰+decode, ρ × 층별 중앙 norm, w_v3)
- 읽기 전용 입력: `P9/hidden_v2/`, `P9/steer_v3/` (현경 산출물). **여기에는 아무것도 쓰지 않음**
- 결과 저장: `내 드라이브/11주차_A축_실험(ldy)/A0_headroom/`

| 행동 (arm) | 층 | 방향 | ρ | 상태 |
|---|---|---|---|---|
| base | – | – | – | 재생성 (기존 `gen_v3_base`와 대조) |
| late_ret_r0.06 | L14–18 | +w | 0.06 | 재생성 (기존 `exp2_P9_r0.06`과 대조 = 훅 검증) |
| late_acc_r0.03 / 0.06 | L14–18 | −w | 0.03 / 0.06 | 신규 |
| early_ret_r0.03 / 0.06 | L8–12 | +w_early | 0.03 / 0.06 | 신규 |
| early_acc_r0.03 / 0.06 | L8–12 | −w_early | 0.03 / 0.06 | 신규 |

**실행 순서:** A0-0 (버전 고정 → 세션 다시 시작) → A0-1 ~ A0-6. A100 기준 생성 약 10~15분.

In [1]:
# ===== A0-0. 버전 고정 (원 실행과 동일) — 실행 후 런타임 > 세션 다시 시작, 그다음 A0-1부터 =====
!pip -q install "transformers==5.17.0" "accelerate==1.14.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.0 MB/s eta 0:00:00


In [2]:
# ===== A0-1. 환경·경로 =====
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"          # CUDA 초기화 전에 설정 (원 노트북과 동일)
import sys, json, re, time, hashlib, importlib, datetime as dt
from collections import Counter
import numpy as np, pandas as pd, torch, transformers
from google.colab import drive; drive.mount("/content/drive")
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False; torch.backends.cudnn.benchmark = False
assert transformers.__version__ == "5.17.0", f"transformers {transformers.__version__} — A0-0 실행 후 세션 다시 시작"

P9 = "/content/drive/MyDrive/11주차_A축_실험(ldy)/P9/"         # 읽기 전용 (현경 산출물)
SV3   = P9 + "steer_v3/"
OUT_A = "/content/drive/MyDrive/11주차_A축_실험(ldy)/A0_headroom/"  # 결과 저장 위치
os.makedirs(OUT_A, exist_ok=True)
assert not OUT_A.startswith(P9), "결과를 공유 폴더에 쓰지 않음"
for f in ["hidden_v2/p8s_behavior_v2.csv", "hidden_v2/p8s_v2_h_out.npy", "steer_v3/env/row_manifest.parquet",
          "steer_v3/env/manifest_cell1.json", "steer_v3/code/p8s_prompt.py", "steer_v3/labels_v3.parquet",
          "steer_v3/cb_v3.parquet", "steer_v3/w_v3.npz", "steer_v3/gen_v3_base.parquet", "steer_v3/exp2_P9_r0.06.parquet"]:
    assert os.path.exists(P9 + f), f"없음: {P9 + f} (P9 바로가기 위치 확인)"

def md5(p, chunk=1 << 24):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

ENV = dict(torch=torch.__version__, transformers=transformers.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, date=dt.datetime.now().isoformat())
print(ENV)

Mounted at /content/drive
{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'gpu': 'NVIDIA A100-SXM4-40GB', 'date': '2026-09-29T01:27:41.938758'}


In [3]:
# ===== A0-2. 데이터·라벨·방향 벡터 (CPU) =====
sys.path.insert(0, SV3 + "code"); import p8s_prompt as PP; importlib.reload(PP)
assert md5(SV3 + "code/p8s_prompt.py") == "20959a4c34203ea0866d030baf23595d", "프롬프트 모듈 불일치"
M1  = json.load(open(SV3 + "env/manifest_cell1.json")); REV = M1["model_revision"]

m   = pd.read_csv(P9 + "hidden_v2/p8s_behavior_v2.csv")
man = pd.read_parquet(SV3 + "env/row_manifest.parquet")
assert (man.question.values == m.question.values).all() and (man.ctx_label.values == m.ctx_label.values).all()
m["rid"], m["fold"], m["n_prompt_tok"] = np.arange(len(m)), man.fold.values, man.n_prompt_tok.values
isCm = (m.ctx_label == "C-").values

Lb = pd.read_parquet(SV3 + "labels_v3.parquet")                  # ret_x / acc_x (baseline 행동), kside_v3
assert (Lb.rid.values == m.rid.values).all()
cb = pd.read_parquet(SV3 + "cb_v3.parquet")
kv = m.question.map(dict(zip(cb.question, cb.kside_v3))).str.replace("−", "-").values     # 'K+'/'K-'
CELL = np.array([k + c for k, c in zip(kv, m.ctx_label.str.replace("−", "-").values)])   # 'K+C-' 등
print("셀 분포:", pd.Series(CELL).value_counts().to_dict())

# 방향 벡터: late(L14–18)는 저장된 w_v3, early(L8–12)는 같은 절차로 새로 계산
LATE, EARLY = list(range(14, 19)), list(range(8, 13))
Z = np.load(SV3 + "w_v3.npz")
W = {k: {L: Z[f"w_f{k}_L{L}"].astype(np.float32) for L in LATE} for k in range(5)}

H = np.load(P9 + "hidden_v2/p8s_v2_h_out.npy", mmap_mode="r")
HL = {L: np.asarray(H[:, L]).astype(np.float32) for L in EARLY + LATE}
PR = Lb.prop.values; ret, acc = (Lb.ret_x == 1).values, (Lb.acc_x == 1).values; isC = (Lb.ctx_label == "C-").values

def prop_bal_diff(X, pos, neg, min_n=3):                           # Steering_develop Cell 9와 동일
    ds = []
    for q in np.unique(PR[pos | neg]):
        i = PR == q
        if (pos & i).sum() >= min_n and (neg & i).sum() >= min_n: ds.append(X[pos & i].mean(0) - X[neg & i].mean(0))
    d = np.mean(ds, 0); return d / np.linalg.norm(d)

chk = []
for k in range(5):
    tr = (Lb.fold != k).values & isC
    for L in EARLY: W[k][L] = prop_bal_diff(HL[L], tr & ret, tr & acc)
    chk.append(min(float(prop_bal_diff(HL[L], tr & ret, tr & acc) @ W[k][L]) for L in LATE))
print("절차 검증: 재계산 w vs 저장 w_v3 (L14–18) 최소 cos =", np.round(chk, 5))
assert min(chk) > 0.9999, "w 재계산 절차가 원본과 다름"
MED = {k: {L: float(np.median(np.linalg.norm(HL[L][(m.fold != k).values], axis=1))) for L in EARLY + LATE} for k in range(5)}
print("MED fold0:", {L: round(v, 2) for L, v in MED[0].items()})
print("cos(w_L12, w_L14) fold0:", round(float(W[0][12] @ W[0][14]), 3), "(층이 달라 참고용)")
np.savez(OUT_A + "w_early_L8-12.npz", **{f"w_f{k}_L{L}": W[k][L] for k in range(5) for L in EARLY})
del HL

셀 분포: {'K+C+': 346, 'K+C-': 346, 'K-C+': 266, 'K-C-': 266}
절차 검증: 재계산 w vs 저장 w_v3 (L14–18) 최소 cos = [1. 1. 1. 1. 1.]
MED fold0: {8: 4.67, 9: 5.76, 10: 6.06, 11: 6.13, 12: 6.57, 14: 8.15, 15: 8.71, 16: 9.62, 17: 10.81, 18: 12.21}
cos(w_L12, w_L14) fold0: 0.266 (층이 달라 참고용)


In [4]:
# ===== A0-3. 모델·생성기·채점기 (GPU) =====
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
if not os.environ.get("HF_TOKEN"):
    from getpass import getpass; login(token=getpass("HF read token: ").strip())
tok = AutoTokenizer.from_pretrained(PP.MODEL_ID, revision=REV)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(PP.MODEL_ID, revision=REV, dtype=torch.bfloat16,
                                             device_map="auto", attn_implementation="sdpa").eval()
EOS_IDS = sorted({tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None})
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
prompts = [tok.apply_chat_template(PP.build_messages(r.question, r.ctx_text), tokenize=False, add_generation_prompt=True)
           for r in m.itertuples()]
assert all(hashlib.md5(p.encode()).hexdigest() == h for p, h in zip(prompts, man.prompt_md5)), "프롬프트 해시 불일치"
m["prompt"] = prompts
BS = 16
FOLD_ROWS = {k: m[m.fold == k].sort_values(["n_prompt_tok", "rid"], kind="stable").rid.values for k in range(5)}

class SteerA:                                     # Steering_develop Steer2와 같은 연산, 층 집합만 확장
    def __init__(self, layers):
        self.layers, self.vec, self.scale, self.mask, self.calls = list(layers), None, {}, None, {}
        self.h = [model.model.layers[L].register_forward_hook(self._mk(L)) for L in self.layers]
    def _mk(self, L):
        def hook(mod, args, out):
            if self.vec is None or L not in self.vec: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            d = (self.scale[L] * self.vec[L]).to(h.dtype)
            self.calls[L] = self.calls.get(L, 0) + 1
            if h.shape[1] > 1: h += self.mask[:, :, None].to(h.dtype) * d          # prefill: 전체 비패딩 토큰 (P9)
            else: h[:, -1, :] += d                                                 # decode
            return out
        return hook
    def set(self, vec=None, scale=None):
        self.vec = None if vec is None else {L: torch.as_tensor(v, device=model.device, dtype=model.dtype) for L, v in vec.items()}
        self.scale, self.calls = (scale or {}), {}
    def remove(self):
        for x in self.h: x.remove()
st = SteerA(EARLY + LATE)

@torch.no_grad()
def gen_fold(k):
    out, rids = {}, FOLD_ROWS[k]
    for s in range(0, len(rids), BS):
        b = rids[s:s + BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        st.mask = enc.attention_mask.clone()
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row)
            out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n)
    return out

# 채점: Steering_develop Cell 6·10의 trunc_v3 / score_v3와 동일
ABBR = r"(?<!\bDr)(?<!\bMr)(?<!\bMs)(?<!\bMrs)(?<!\bSt)(?<!\bJr)(?<!\bSr)(?<!\b[A-Z])"
def trunc_v3(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    if len(s.split()) > 8: s = re.split(ABBR + r"\.\s+(?=[A-Z])", s)[0]
    return re.sub(r"[.,;:\s]+$", "", s.strip().strip('"\'')).strip()
def rep3(t):
    w = str(t).split(); g = Counter(tuple(w[i:i + 3]) for i in range(len(w) - 2))
    return int(bool(g) and max(g.values()) >= 3)
gl_row = [json.loads(g) if isinstance(g, str) else list(g) for g in m.gold]
cbq = dict(zip(cb.question, cb.cb_greedy_v3))
def score_v3(d):
    p = [PP.norm(trunc_v3(x)) for x in d.raw]
    s, c = m.answer_surface.map(PP.norm).values, m.question.map(cbq).map(PP.norm).values
    d["pred"] = [trunc_v3(x) for x in d.raw]
    d["em"] = [int(any(pi == PP.norm(g) for g in gg)) for pi, gg in zip(p, gl_row)]
    d["acc_x"] = [int(si != "" and pi == si) for pi, si in zip(p, s)]
    d["ret_x"] = [int(ci != "" and ci != si and pi == ci) for pi, si, ci in zip(p, s, c)]
    d["hit_limit"] = (d.n_tok >= PP.MAX_NEW_TOKENS).astype(int); d["rep3"] = d.raw.map(rep3)
    return d

def run_arm(name, layers=None, sign=0.0, rho=0.0):
    f = OUT_A + f"gen_{name}.parquet"
    if os.path.exists(f): return pd.read_parquet(f)                     # 체크포인트 재사용
    rows, t0 = [], time.time()
    for k in range(5):
        if layers is None: st.set(None)
        else: st.set({L: sign * W[k][L] for L in layers}, {L: rho * MED[k][L] for L in layers})
        o = gen_fold(k)
        if layers is not None: assert all(st.calls.get(L, 0) > 0 for L in layers), "훅 미호출"
        rows += [dict(rid=r, raw=x, n_tok=n, fold=k) for r, (x, n) in o.items()]
    st.set(None)
    d = score_v3(pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)); d["arm"] = name
    d.to_parquet(f, index=False)
    print(f"{name:16s} {time.time() - t0:4.0f}s  C− {d.em.values[isCm].mean():.4f}  C+ {d.em.values[~isCm].mean():.4f}  "
          f"hit {d.hit_limit.mean():.3f}  rep3 {d.rep3.mean():.3f}")
    return d
print("준비 완료:", torch.cuda.get_device_name(0))

HF read token: ··········


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

준비 완료: NVIDIA A100-SXM4-40GB


In [5]:
# ===== A0-4. 재현 검증 — 이게 통과해야 새 arm 결과를 믿을 수 있음 =====
ARM = {}
ARM["base"] = run_arm("base")
ARM["late_ret_r0.06"] = run_arm("late_ret_r0.06", LATE, +1.0, 0.06)
ref_b, ref_s = pd.read_parquet(SV3 + "gen_v3_base.parquet"), pd.read_parquet(SV3 + "exp2_P9_r0.06.parquet")
mb = int((ARM["base"].raw.values != ref_b.raw.values).sum()); ms = int((ARM["late_ret_r0.06"].raw.values != ref_s.raw.values).sum())
print(f"문자열 불일치  base {mb}/1224 | late_ret_r0.06 {ms}/1224  (현경 배치 민감도 실험 기준 ~1% 이내면 허용)")
print(f"EM 비교  base {ARM['base'].em.mean():.4f} vs 기존 {ref_b.em.mean():.4f} | "
      f"steer {ARM['late_ret_r0.06'].em.mean():.4f} vs 기존 {ref_s.em.mean():.4f}")
assert mb <= 15 and ms <= 20, "재현 실패 — 버전·프롬프트·훅 확인 후 진행"
json.dump({**ENV, "repro_mismatch_base": mb, "repro_mismatch_late_ret": ms, "model_revision": REV},
          open(OUT_A + "env_repro.json", "w"), indent=1, ensure_ascii=False)

문자열 불일치  base 0/1224 | late_ret_r0.06 0/1224  (현경 배치 민감도 실험 기준 ~1% 이내면 허용)
EM 비교  base 0.5310 vs 기존 0.5310 | steer 0.5997 vs 기존 0.5997


In [6]:
# ===== A0-5. 새 행동 생성 (조건별 체크포인트, A100 조건당 약 1분) =====
PLAN = [("late_acc_r0.03", LATE, -1.0, 0.03), ("late_acc_r0.06", LATE, -1.0, 0.06),
        ("early_ret_r0.03", EARLY, +1.0, 0.03), ("early_ret_r0.06", EARLY, +1.0, 0.06),
        ("early_acc_r0.03", EARLY, -1.0, 0.03), ("early_acc_r0.06", EARLY, -1.0, 0.06)]
for name, layers, sign, rho in PLAN: ARM[name] = run_arm(name, layers, sign, rho)

In [7]:
# ===== A0-6. 여지 분석 (CPU) =====
names = list(ARM); E = np.stack([ARM[n].em.values for n in names], 1).astype(float)   # (1224, arm)
base = E[:, 0]; ALW = E[:, names.index("late_ret_r0.06")]
stable = {n: bool(ARM[n].hit_limit.mean() <= .05 and ARM[n].rep3.mean() <= .05) for n in names}
q = m.question.values; uq = np.unique(q); qidx = {u: np.where(q == u)[0] for u in uq}
rs_ = np.random.RandomState(0); BOOT = [np.concatenate([qidx[u] for u in rs_.choice(uq, len(uq))]) for _ in range(2000)]
def ci(v, msk=None):
    msk = np.ones(len(v), bool) if msk is None else msk
    return np.percentile([v[i][msk[i]].mean() for i in BOOT], [2.5, 97.5])
fold = m.fold.values; cells = ["K+C+", "K+C-", "K-C+", "K-C-"]

def cross_fit(groups):
    # 그룹(예: K×C 셀)별로 다른 fold에서 평균 EM이 가장 높은 arm을 골라 held-out fold에 적용 (안정 arm만)
    ok = [j for j, n in enumerate(names) if stable[n]]; pol = np.zeros(len(base)); pick = {}
    for k in range(5):
        tr, te = fold != k, fold == k
        for g in np.unique(groups):
            gi = groups == g; j = ok[int(np.argmax(E[tr & gi][:, ok].mean(0)))]
            pol[te & gi] = E[te & gi, j]; pick[(k, g)] = names[j]
    return pol, pick

POL = {"no-op": base}
POL.update({f"always:{n}": E[:, j] for j, n in enumerate(names) if n != "base"})
POL["oracle: retain-only (base|late_ret)"] = np.maximum(base, ALW)
late = [names.index(n) for n in names if n == "base" or n.startswith("late")]
POL["oracle: late arms"] = E[:, late].max(1)
POL["oracle: all arms"] = E[:, [j for j, n in enumerate(names) if stable[n]]].max(1)
POL["state-oracle: C only (cross-fit)"], pick_c = cross_fit(m.ctx_label.str.replace("−", "-").values)
POL["state-oracle: K only (cross-fit)"], pick_k = cross_fit(kv)
POL["state-oracle: K×C (cross-fit)"], pick_kc = cross_fit(CELL)

rows = []
for n, v in POL.items():
    d, da = v - base, v - ALW
    r = dict(policy=n, EM=v.mean(), dEM=d.mean(), dC_minus=d[isCm].mean(), dC_plus=d[~isCm].mean(),
             vs_always_ret=da.mean(), vs_lo=ci(da)[0], vs_hi=ci(da)[1])
    r.update({f"d_{c}": d[CELL == c].mean() for c in cells}); rows.append(r)
T = pd.DataFrame(rows); T.to_csv(OUT_A + "A0_policies.csv", index=False)
pd.set_option("display.width", 250)
print("[arm 안정성]", {n: ("OK" if s else "불안정") for n, s in stable.items()})
print(T.round(4).to_string(index=False))

C_ = pd.DataFrame([{**{"arm": n}, **{c: (E[:, j] - base)[CELL == c].mean() for c in cells},
                    "C-": (E[:, j] - base)[isCm].mean(), "C+": (E[:, j] - base)[~isCm].mean()} for j, n in enumerate(names)])
C_.to_csv(OUT_A + "A0_arm_by_cell.csv", index=False)
print("\n[arm별 ΔEM vs base, 셀별]"); print(C_.round(4).to_string(index=False))
print("\n[K×C state-oracle이 고른 행동 (fold별)]")
print(pd.DataFrame([{"fold": k, **{c: pick_kc.get((k, c)) for c in cells}} for k in range(5)]).to_string(index=False))

pd.DataFrame(E.astype(int), columns=names).assign(rid=m.rid, cell=CELL, fold=fold).to_parquet(OUT_A + "A0_outcome_table.parquet", index=False)
kc = T.set_index("policy").loc["state-oracle: K×C (cross-fit)"]
verdict = ("여지 있음 → 2단계(router) 진행" if kc.vs_lo > 0 else
           "K×C를 완벽히 알아도 Always-retain과 유의차 없음 → 현재 행동 집합으로는 방향 A 이득 제한적")
json.dump({"verdict": verdict, "state_oracle_KxC_vs_always": [float(kc.vs_always_ret), float(kc.vs_lo), float(kc.vs_hi)],
           "oracle_all_vs_always": float(T.set_index("policy").loc["oracle: all arms"].vs_always_ret),
           "stable": stable, "K_label": "kside_v3", "setting": "p8s-v2, P9 all+decode, rho*median||h||"},
          open(OUT_A + "A0_summary.json", "w"), indent=1, ensure_ascii=False)
print("\n판정:", verdict, "| 저장:", OUT_A)

[arm 안정성] {'base': 'OK', 'late_ret_r0.06': 'OK', 'late_acc_r0.03': 'OK', 'late_acc_r0.06': 'OK', 'early_ret_r0.03': 'OK', 'early_ret_r0.06': 'OK', 'early_acc_r0.03': 'OK', 'early_acc_r0.06': 'OK'}
                             policy     EM     dEM  dC_minus  dC_plus  vs_always_ret   vs_lo   vs_hi  d_K+C+  d_K+C-  d_K-C+  d_K-C-
                              no-op 0.5310  0.0000    0.0000   0.0000        -0.0686 -0.0858 -0.0515  0.0000  0.0000  0.0000  0.0000
              always:late_ret_r0.06 0.5997  0.0686    0.1356   0.0016         0.0000  0.0000  0.0000  0.0058  0.1994 -0.0038  0.0526
              always:late_acc_r0.03 0.4804 -0.0507   -0.0899  -0.0114        -0.1193 -0.1397 -0.0972 -0.0145 -0.1416 -0.0075 -0.0226
              always:late_acc_r0.06 0.4395 -0.0915   -0.1650  -0.0180        -0.1601 -0.1830 -0.1364 -0.0347 -0.2659  0.0038 -0.0338
             always:early_ret_r0.03 0.5547  0.0237    0.0474   0.0000        -0.0449 -0.0605 -0.0302  0.0000  0.0665  0.0000  0.0226
     

In [8]:
# ===== A0-5b. late_ret ρ0.03/0.09/0.12 직접 생성 + 현경 exp2 대조 =====
ARM = {}; CMP = {}
for r in ["0.03", "0.09", "0.12"]:
    name = f"late_ret_r{r}"
    ARM[name] = run_arm(name, LATE, +1.0, float(r))
    ref = pd.read_parquet(SV3 + f"exp2_P9_r{r}.parquet")
    mis = int((ARM[name].raw.values != ref.raw.values).sum())
    CMP[name] = dict(mismatch=mis, em_mine=float(ARM[name].em.mean()), em_hk=float(ref.em.mean()))
    print(f"{name}: 문자열 불일치 {mis}/1224 | EM {CMP[name]['em_mine']:.4f} vs 현경 {CMP[name]['em_hk']:.4f}")
json.dump(CMP, open(OUT_A + "repro_late_ret_extra.json", "w"), indent=1)

from google.colab import drive
drive.flush_and_unmount()   # Drive에 확실히 기록 (A1 세션에서 바로 보이도록)

late_ret_r0.03     56s  C− 0.3431  C+ 0.8023  hit 0.000  rep3 0.000
late_ret_r0.03: 문자열 불일치 0/1224 | EM 0.5727 vs 현경 0.5727
late_ret_r0.09     54s  C− 0.4265  C+ 0.7745  hit 0.000  rep3 0.000
late_ret_r0.09: 문자열 불일치 0/1224 | EM 0.6005 vs 현경 0.6005
late_ret_r0.12     82s  C− 0.4216  C+ 0.7320  hit 0.002  rep3 0.002
late_ret_r0.12: 문자열 불일치 0/1224 | EM 0.5768 vs 현경 0.5768
